# Tunable compliance — emergent grasps

This notebook analyses two hand experiments that demonstrate **emergent grasp behaviours** arising from
tunable virtual stiffness (VMC):

1. **In-hand manipulation** — asymmetric fingertip stiffness is used to reorient an object already held in the hand, without any explicit trajectory planning for the fingers.
2. **Dynamic grasping** — the hand closes on a moving object under soft, stiff, and adaptive stiffness schedules, showing that adaptive compliance achieves a gentler initial contact followed by a firm final grip.

---
## In-hand manipulation — asymmetric fingertip stiffness

### Experimental setup

The UR5 is held fixed at `UR5_POSE_INHAND`. The hand closes to a **PC1 grasp** (Santello et al. 1998 — first principal component of human grasp data) and three stiffness configurations are applied sequentially:

| Phase | Side A (pinky + ring) | Side B (thumb + index + middle) |
|-------|----------------------|----------------------------------|
| **Uniform** | `K_UNIFORM` | `K_UNIFORM` |
| **Asym A**  | `K_LOW` (soft) | `K_HIGH` (stiff) |
| **Asym B**  | `K_HIGH` (stiff) | `K_LOW` (soft) |

### What we want to prove

- **Asymmetric stiffness drives asymmetric deformation**: soft-side fingers deform more (larger tip displacement from the PC1 reference), stiff-side fingers resist — producing a net rolling/tilting moment on the grasped object.
- **The effect is reversible and side-selective**: flipping the stiffness assignment (Asym A → Asym B) mirrors the displacement pattern — a *seesaw effect*.
- **Stiffness and displacement are inversely correlated**: confirming that VMC tip stiffness directly controls contact compliance, not just joint angles.
- **Contact forces follow the complementary pattern**: stiff fingers carry more force, soft fingers less — consistent with a differential contact geometry.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, sys
from pathlib import Path

sys.path.insert(0, os.path.join('../..'))
plt.style.use(os.path.join('../..', 'plot_config.mplstyle'))

FINGERTIPS   = ['thumb', 'index', 'middle', 'ring', 'pinky']
PHASES       = ['uniform', 'asym_a', 'asym_b']
PHASE_LABELS = {'uniform': 'Uniform', 'asym_a': 'Asym A', 'asym_b': 'Asym B'}
PHASE_COLORS = {'uniform': '#56B4E9', 'asym_a': '#E69F00', 'asym_b': '#D55E00'}

SIDE_A = ['pinky', 'ring']             # soft in Asym A, stiff in Asym B
SIDE_B = ['thumb', 'index', 'middle']  # stiff in Asym A, soft in Asym B

OUTPUT_INHAND = os.path.join('outputs', 'inhand_manipulation')
os.makedirs(OUTPUT_INHAND, exist_ok=True)


def load_inhand():
    path = Path(OUTPUT_INHAND) / 'inhand_run.csv'
    return pd.read_csv(path) if path.exists() else None


df = load_inhand()
if df is None:
    print('No data found — run inhand_manipulation.py first.')
else:
    print(f'Loaded {len(df)} rows | phases: {df["phase"].value_counts().to_dict()}')

### Mean tip displacement per finger

Mean Euclidean displacement of each fingertip from its PC1 reference position, grouped by phase.
Soft-side fingers should show larger displacements in the corresponding asymmetric phase.
Error bars show one standard deviation over the recording window.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(FINGERTIPS))
width = 0.25

for i, phase in enumerate(PHASES):
    means, stds = [], []
    for f in FINGERTIPS:
        if df is not None:
            rows = df[df['phase'] == phase]
            mag  = np.sqrt(rows[f'disp_{f}_x_m']**2 +
                           rows[f'disp_{f}_y_m']**2 +
                           rows[f'disp_{f}_z_m']**2) * 1e3
            means.append(float(mag.mean()))
            stds.append(float(mag.std()))
        else:
            means.append(0.0); stds.append(0.0)
    ax.bar(x + i * width, means, width, yerr=stds, capsize=3,
           label=PHASE_LABELS[phase], color=PHASE_COLORS[phase])

ax.set_xticks(x + width)
ax.set_xticklabels(FINGERTIPS)
ax.set_xlabel('Finger')
ax.set_ylabel(r'Mean $\|\Delta p\|$ [mm]')
ax.set_title('Tip displacement from PC1 reference')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_INHAND, 'inhand_tip_displacement.pdf'), bbox_inches='tight')
plt.show()

### Seesaw effect: soft vs stiff side

Displacement averaged over all fingers on each side. Dashed lines show the Uniform baseline for each side.
The asymmetric stiffness should produce a clear reversal between Asym A and Asym B — the *seesaw*:
in Asym A, Side A (pinky+ring) is soft and deforms more; in Asym B, Side B (thumb+index+middle) is soft and the pattern flips.

In [ ]:
def side_stats(df_, phase, fingers):
    if df_ is None:
        return 0.0, 0.0
    rows = df_[df_['phase'] == phase]
    mags = np.concatenate([
        np.sqrt(rows[f'disp_{f}_x_m']**2 +
                rows[f'disp_{f}_y_m']**2 +
                rows[f'disp_{f}_z_m']**2).values * 1e3
        for f in fingers
    ])
    return float(mags.mean()), float(mags.std())


fig, axes = plt.subplots(1, 2, figsize=(8, 4), sharey=True)
fig.suptitle('Seesaw effect: soft-side vs stiff-side displacement')

for ax, phase in zip(axes, ['asym_a', 'asym_b']):
    soft_side   = SIDE_A if phase == 'asym_a' else SIDE_B
    stiff_side  = SIDE_B if phase == 'asym_a' else SIDE_A
    soft_label  = 'Side A\n(pinky+ring)'      if phase == 'asym_a' else 'Side B\n(thumb+idx+mid)'
    stiff_label = 'Side B\n(thumb+idx+mid)'   if phase == 'asym_a' else 'Side A\n(pinky+ring)'

    m_soft,  s_soft  = side_stats(df, phase, soft_side)
    m_stiff, s_stiff = side_stats(df, phase, stiff_side)
    m_uni_a, _       = side_stats(df, 'uniform', SIDE_A)
    m_uni_b, _       = side_stats(df, 'uniform', SIDE_B)
    m_uni_soft  = m_uni_a if phase == 'asym_a' else m_uni_b
    m_uni_stiff = m_uni_b if phase == 'asym_a' else m_uni_a

    ax.bar([soft_label, stiff_label], [m_soft, m_stiff],
           yerr=[s_soft, s_stiff], capsize=4,
           color=[PHASE_COLORS[phase], '#AAAAAA'])
    ax.axhline(m_uni_soft,  color=PHASE_COLORS['uniform'], ls='--', lw=1.2,
               alpha=0.8, label='Uniform ref (soft side)')
    ax.axhline(m_uni_stiff, color=PHASE_COLORS['uniform'], ls=':',  lw=1.2,
               alpha=0.8, label='Uniform ref (stiff side)')
    ax.set_title(PHASE_LABELS[phase])
    ax.set_ylabel(r'Mean $\|\Delta p\|$ [mm]')
    ax.legend(fontsize=7)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_INHAND, 'inhand_seesaw.pdf'), bbox_inches='tight')
plt.show()

### Compliance–displacement correlation

Each point is one finger at one timestep. Stiffness and tip displacement should be **inversely correlated**:
low K → large deformation, high K → small deformation.
This validates that the VMC tip spring directly governs contact compliance.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

if df is not None:
    for phase in PHASES:
        rows = df[df['phase'] == phase]
        first = True
        for f in FINGERTIPS:
            k   = rows[f'k_{f}_Npm'].values
            mag = np.sqrt(rows[f'disp_{f}_x_m']**2 +
                          rows[f'disp_{f}_y_m']**2 +
                          rows[f'disp_{f}_z_m']**2).values * 1e3
            ax.scatter(k, mag, s=4, alpha=0.3, color=PHASE_COLORS[phase],
                       label=PHASE_LABELS[phase] if first else '')
            first = False

handles, labels = ax.get_legend_handles_labels()
ax.legend(dict(zip(labels, handles)).values(), dict(zip(labels, handles)).keys(),
          markerscale=3)
ax.set_xlabel('Applied tip stiffness $K$ [N/m]')
ax.set_ylabel(r'Tip displacement $\|\Delta p\|$ [mm]')
ax.set_title('Compliance–displacement correlation')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_INHAND, 'inhand_k_vs_disp.pdf'), bbox_inches='tight')
plt.show()

### Contact force magnitude per finger

Mean estimated contact force magnitude (1st-order stiffness model) per finger and phase.
Stiff-side fingers should carry **more** force; soft-side fingers **less** — the complementary side of the seesaw.
This force asymmetry is what drives the object to roll toward the soft side.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(FINGERTIPS))
width = 0.25

for i, phase in enumerate(PHASES):
    means, stds = [], []
    for f in FINGERTIPS:
        col = f'force_1st_{f}_mag_N'
        if df is not None and col in df.columns:
            rows = df[df['phase'] == phase]
            means.append(float(rows[col].mean()))
            stds.append(float(rows[col].std()))
        else:
            means.append(0.0); stds.append(0.0)
    ax.bar(x + i * width, means, width, yerr=stds, capsize=3,
           label=PHASE_LABELS[phase], color=PHASE_COLORS[phase])

ax.set_xticks(x + width)
ax.set_xticklabels(FINGERTIPS)
ax.set_xlabel('Finger')
ax.set_ylabel(r'Mean $|F_{tip}|$ [N]')
ax.set_title('Contact force magnitude (1st-order model)')
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_INHAND, 'inhand_force.pdf'), bbox_inches='tight')
plt.show()

### Effective tip stiffness eigenvalues

Eigenvalues of the 3×3 tip stiffness matrix (1st-order model) per finger, shown as box plots split by phase.
The **minimum eigenvalue** reflects the most compliant direction at the fingertip — this should drop sharply
for soft-side fingers in the corresponding asymmetric phase, confirming that VMC reshapes the full
stiffness ellipsoid, not just the isotropic magnitude.

In [ ]:
eig_ok = df is not None and 'stiff_1st_thumb_eig_0_Npm' in df.columns

if eig_ok:
    fig, axes = plt.subplots(1, len(FINGERTIPS), figsize=(14, 4), sharey=True)
    fig.suptitle('Tip stiffness eigenvalues by phase (1st-order model)')

    for ax, finger in zip(axes, FINGERTIPS):
        data, tick_labels = [], []
        tick_colors = []
        for phase in PHASES:
            mask = df['phase'] == phase
            for k in range(3):
                col = f'stiff_1st_{finger}_eig_{k}_Npm'
                data.append(df.loc[mask, col].dropna().values)
                tick_labels.append(f'{PHASE_LABELS[phase]}\n$\\lambda_{k+1}$')
                tick_colors.append(PHASE_COLORS[phase])

        bp = ax.boxplot(data, labels=tick_labels, patch_artist=True,
                        showfliers=False, widths=0.6)
        for patch, color in zip(bp['boxes'], tick_colors):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)

        ax.set_title(finger)
        ax.tick_params(axis='x', labelsize=6)
        ax.grid(True, axis='y', alpha=0.3)
        if ax is axes[0]:
            ax.set_ylabel('Eigenvalue [N/m]')

    fig.tight_layout()
    fig.savefig(os.path.join(OUTPUT_INHAND, 'inhand_stiffness_eigenvalues.pdf'), bbox_inches='tight')
    plt.show()
else:
    print('Stiffness eigenvalue columns not found — run inhand_manipulation.py first.')

---
## Dynamic grasping — soft vs stiff vs adaptive

### Experimental setup

The UR5 moves the hand along **+X** at `APPROACH_SPEED = 0.05 m/s` from `UR5_POSE_BOTTLE_START`.
At `CLOSE_DISTANCE = 0.15 m` the hand closes to the PC1 pose. Three stiffness schedules are compared:

| Condition | K during approach | K after close |
|-----------|-------------------|---------------|
| **Soft**     | `K_SOFT`   | `K_SOFT` (remains compliant)        |
| **Stiff**    | `K_STIFF`  | `K_STIFF` (rigid throughout)        |
| **Adaptive** | `K_SOFT`   | ramps to `K_STIFF` after `SOFT_DURATION` |

### What we want to prove

- **Soft** contact reduces impact forces at close but yields a weak final grip (fingers keep moving).
- **Stiff** contact achieves a firm grip quickly but risks high impact forces on close.
- **Adaptive** stiffness combines both advantages: the initial soft phase absorbs contact, then the stiffness ramp secures the grasp — the finger trajectory stabilises faster than soft, with less initial jerk than stiff.

In [ ]:
OUTPUT_GRASP = os.path.join('outputs', 'dynamic_grasp')
os.makedirs(OUTPUT_GRASP, exist_ok=True)

CONDITIONS = ['soft', 'stiff', 'adaptive']
COLORS_DG  = {'soft': '#56B4E9', 'stiff': '#D55E00', 'adaptive': '#009E73'}


def load_dynamic(condition):
    path = Path(OUTPUT_GRASP) / f'dynamic_grasp_{condition}.csv'
    return pd.read_csv(path) if path.exists() else None


N_GRID = 300


def interp_trace(arr, n=N_GRID):
    x = np.linspace(0, 1, len(arr))
    return np.interp(np.linspace(0, 1, n), x, arr)


fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))

for cond, color in COLORS_DG.items():
    df_dg = load_dynamic(cond)
    if df_dg is None or df_dg.empty:
        continue
    closed = df_dg[df_dg['phase'] != 'open']
    if closed.empty:
        continue
    T = np.linspace(0, len(closed) / 30, N_GRID)  # ~30 Hz logging

    # Index MCP angle (motor 7) — shows finger closure trajectory
    axes[0].plot(T, np.rad2deg(interp_trace(closed['q_7'].to_numpy())),
                 color=color, label=cond.capitalize())

    # Applied tip stiffness — shows the adaptive ramp
    axes[1].plot(T, interp_trace(closed['k_tip_Npm'].to_numpy()),
                 color=color, label=cond.capitalize())

axes[0].set_xlabel('Time after close [s]')
axes[0].set_ylabel('Index MCP angle [deg]')
axes[0].set_title('Finger closure trajectory')
axes[0].legend()

axes[1].set_xlabel('Time after close [s]')
axes[1].set_ylabel('$K_{tip}$ [N/m]')
axes[1].set_title('Stiffness schedule')
axes[1].legend()

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_GRASP, 'dynamic_grasp_comparison.pdf'), bbox_inches='tight')
plt.show()